# Đánh giá kiến trúc backend trên Kaggle CPU

Chọn **Accelerator: None**, bật Internet để chuẩn bị dependency. Thêm Dataset riêng chứa `cinema-phase2-source.tar.gz` hoặc source đã được Kaggle tự giải nén. Ô đầu tự nhận diện và chép source vào `/kaggle/working`; không cần nén lại Dataset. Chạy các ô theo thứ tự trong cùng phiên CPU. Kết quả smoke không thay thế ma trận đo chính gồm 144 lượt.


In [ ]:
from pathlib import Path
import hashlib, os, shutil, subprocess, sys, tarfile, tempfile


def source_roots(directory):
    required = ['app/main.py', 'alembic.ini', 'requirements.txt',
                'experiments/phase2/bootstrap.py', 'experiments/phase2/requirements.lock.txt']
    return sorted({path.parents[2] for path in directory.rglob('bootstrap.py')
                   if all((path.parents[2] / name).is_file() for name in required)})


def prepare_source(input_root, working_root):
    input_root, working_root = Path(input_root), Path(working_root)
    archives = sorted(input_root.rglob('cinema-phase2-source.tar.gz'))
    if len(archives) > 1:
        raise RuntimeError('Có nhiều gói source; chỉ gắn một Dataset: ' + str(archives))
    if archives:
        source = archives[0]
        checksum = hashlib.sha256(source.read_bytes()).hexdigest()
        mode = 'Tệp nén'
    else:
        roots = source_roots(input_root)
        if len(roots) != 1:
            raise RuntimeError('Cần đúng một bộ source. Chọn Add Input để gắn Dataset '
                               'chứa gói source hoặc source đã giải nén. Các bộ tìm thấy: ' + str(roots))
        source = roots[0]
        digest = hashlib.sha256()
        for path in sorted(source.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts and path.suffix != '.pyc':
                digest.update(path.relative_to(source).as_posix().encode() + b'\0')
                digest.update(hashlib.sha256(path.read_bytes()).digest())
        checksum = digest.hexdigest()
        mode = 'Source đã được Kaggle giải nén'
    working_root.mkdir(parents=True, exist_ok=True)
    repo = working_root / ('cinema-phase2-' + checksum[:12])
    if not repo.exists():
        # Publish only a complete copy; an interrupted cell can be run again.
        with tempfile.TemporaryDirectory(prefix='phase2-source-', dir=working_root) as temporary:
            staging = Path(temporary) / 'source'
            if archives:
                staging.mkdir()
                with tarfile.open(source) as archive:
                    archive.extractall(staging, filter='data')
            else:
                shutil.copytree(source, staging, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
            roots = source_roots(staging)
            if len(roots) != 1:
                raise RuntimeError('Gói source thiếu cấu trúc backend hợp lệ: ' + str(roots))
            roots[0].rename(repo)
    if repo not in source_roots(repo):
        raise RuntimeError('Thư mục working chưa có source đầy đủ: ' + str(repo))
    print('Input mode:', mode)
    print('Input source:', source)
    print('Source checksum:', checksum)
    print('Repository:', repo)
    return repo


repo = prepare_source('/kaggle/input', '/kaggle/working')


## Chuẩn bị môi trường

Ô này cài PostgreSQL 16, Redis 7 và môi trường Python 3.12 với dependency đã khóa. Trình cài chỉ chạy trong Kaggle; các dịch vụ thử nghiệm được khởi động riêng khi kiểm chứng hoặc đo tải.


In [ ]:
sys.path.insert(0, str(repo))
from experiments.phase2.notebook import run_checked
run_checked([sys.executable, str(repo / 'experiments/phase2/bootstrap.py'), '--install'], cwd=repo)
python = '/kaggle/working/phase2-venv/bin/python'
def run(*arguments):
    run_checked([python, *arguments], cwd=repo)


## Kiểm chứng chức năng

Chạy toàn bộ kiểm thử, bao gồm PostgreSQL qua migration, cache fallback và tranh chấp giữa hai worker. Dừng tại đây nếu kiểm thử không đạt.


In [ ]:
run('-m', 'experiments.phase2.verify')


## Kiểm tra nhanh

Smoke gồm 16 lượt, 10 người dùng, warm-up 5 giây và đo 10 giây. Dùng bộ dữ liệu chuẩn. Mỗi thư mục kết quả chỉ dành cho một lần chạy; không ghi đè kết quả cũ.


In [ ]:
smoke = Path('/kaggle/working/phase2-results-smoke')
run('-m', 'experiments.phase2.runner', '--profile', 'smoke', '--output-root', str(smoke))
run('-m', 'experiments.phase2.report', str(smoke))


## Thí nghiệm chính

Bốn biến thể × bốn kịch bản × ba mức tải × ba lần lặp. Dự kiến 3–4 giờ kể cả chuẩn bị dữ liệu; thời gian thực tế phụ thuộc môi trường. Không đổi source, dependency hoặc cấu hình notebook trong khi chạy. Ô này chỉ chạy tiếp sau khi smoke trả mã thành công.


In [ ]:
full = Path('/kaggle/working/phase2-results-full')
run('-m', 'experiments.phase2.runner', '--profile', 'full', '--output-root', str(full))


## Tổng hợp và xuất kết quả

Nếu phép đo bị ngắt, vẫn có thể chạy ô này riêng để tổng hợp phần đã có; báo cáo giữ trạng thái chưa hoàn tất. Chỉ kết luận Pha 2 khi `primary_complete` là `true`.


In [ ]:
run('-m', 'experiments.phase2.report', str(full))
import json, shutil
summary = json.loads((full / 'summary.json').read_text())
print({k: summary[k] for k in ['expected_runs', 'valid_runs', 'invalid_runs', 'primary_complete']})
print((full / 'report.md').read_text())
shutil.make_archive('/kaggle/working/phase2-results-full', 'gztar', root_dir=full.parent, base_dir=full.name)
